# 18b — Text-to-SQL (Gemini 2.5 Pro + BigQuery)

**AWS-side equivalent:** Nova Pro + Aurora Serverless v2 (`notebooks/aws/18b-text-to-sql.ipynb`)
**GCP service:** Gemini 2.5 Pro structured output → **BigQuery** via the `google-cloud-bigquery` client

Pattern (same shape as the AWS side):

```
Question → schema digest from INFORMATION_SCHEMA.COLUMNS → Gemini 2.5 Pro
         → JSON {sql, warnings} via response_schema
         → validator (SELECT/WITH only, allow-listed tables, forbidden-keyword regex)
         → BigQuery dry-run guard (bytesProcessed cap)
         → BigQuery execute (maximum_bytes_billed cap)
         → rows + grounded answer
```

_As of 2026-05, `google-genai` (`from google import genai`) is the supported Python client for Gemini structured output. Use `response_mime_type='application/json'` together with `response_schema` (Pydantic or dict). `gemini-2.5-pro` is required here — the reasoning headroom matters for joins and aggregations; `2.5-flash` underperforms on this prompt._

> **Production note:** this notebook relies on Application Default Credentials. In production, run the executor as a dedicated service account with **only** `roles/bigquery.dataViewer` (or table-scoped `roles/bigquery.tables.getData`) + `roles/bigquery.jobUser`. The dry-run + bytes-billed cap are belt-and-braces — IAM is still the primary least-privilege boundary.

In [ ]:
# State plumbing — every notebook reads/writes config.json under the 'gcp' namespace.
import json
from pathlib import Path

from google.cloud import bigquery
from google import genai
from google.genai import types as genai_types

config = json.loads(Path('config.json').read_text())
gcp = config.setdefault('gcp', {})
PROJECT_ID = gcp.get('project_id')
REGION = gcp.get('region', 'us-central1')
assert PROJECT_ID, 'Run 01-setup-credentials.ipynb first.'

TEXT2SQL_MODEL = gcp.get('text2sql_model', 'gemini-2.5-pro')
GROUNDING_MODEL = gcp.get('model_id', 'gemini-2.5-flash')

# BigQuery dataset variables — populated by 02-bigquery-graph.ipynb.
BQ_DATASET = gcp.get('bq_dataset')
BQ_LOCATION = gcp.get('bq_location', 'US')
assert BQ_DATASET, 'BigQuery dataset not configured. Run 02-bigquery-graph.ipynb first.'

# Cost guards — every BQ-backed text-to-SQL needs these. Tune in config.json if needed.
DRY_RUN_BYTES_CAP = int(gcp.get('text2sql_dry_run_bytes_cap', 100_000_000))   # 100 MB
EXEC_BYTES_BILLED_CAP = int(gcp.get('text2sql_bytes_billed_cap', 10_000_000))  # 10 MB

print(f'project={PROJECT_ID} region={REGION} text2sql={TEXT2SQL_MODEL} grounding={GROUNDING_MODEL}')
print(f'bq_dataset={BQ_DATASET} bq_location={BQ_LOCATION}')
print(f'dry_run_cap={DRY_RUN_BYTES_CAP} bytes | exec_cap={EXEC_BYTES_BILLED_CAP} bytes')

## Step 1: Open a BigQuery client

ADC is sufficient for the notebook context (Cloud Shell, `gcloud auth application-default login`, or a Workbench instance attached to a service account). No connection string, no Secret Manager pull — IAM is the boundary. The cell just smoke-tests the dataset is reachable.

In [ ]:
bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

# Smoke test: confirm the dataset exists and is readable.
dataset_ref = bigquery.DatasetReference(PROJECT_ID, BQ_DATASET)
ds = bq.get_dataset(dataset_ref)
print(f'✓ BigQuery dataset {ds.full_dataset_id} ({ds.location})')
print(f'  created={ds.created.isoformat() if ds.created else "?"}')

## Step 2: Build a schema digest from `INFORMATION_SCHEMA.COLUMNS`

The digest is what we hand to Gemini — short, deterministic, and limited to the allow-listed tables. BigQuery exposes `INFORMATION_SCHEMA.COLUMNS` per-dataset; we pass the allow-list as a parameterised `UNNEST(@allowed)` instead of string-formatting table names into the SQL (avoids an injection vector even though we control the input).

In [ ]:
# Tables the user is allowed to query through this surface. Anything outside
# this list is rejected by the validator regardless of what the model emits.
ALLOWED_TABLES = gcp.get('text2sql_allowed_tables') or [
    'people', 'teams', 'skills', 'team_memberships', 'person_skills', 'collaborations'
]


def build_schema_digest(client: bigquery.Client, dataset: str, allowed: list[str]) -> str:
    sql = f"""
        SELECT table_name, column_name, data_type
        FROM   `{PROJECT_ID}.{dataset}.INFORMATION_SCHEMA.COLUMNS`
        WHERE  table_name IN UNNEST(@allowed)
        ORDER BY table_name, ordinal_position
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter('allowed', 'STRING', allowed),
        ]
    )
    rows = list(client.query(sql, job_config=job_config).result())
    by_table: dict[str, list[str]] = {}
    for row in rows:
        by_table.setdefault(row['table_name'], []).append(f"{row['column_name']} {row['data_type']}")
    lines = [f'{PROJECT_ID}.{dataset}.{table}({", ".join(cols)})' for table, cols in by_table.items()]
    return '\n'.join(lines)


SCHEMA_DIGEST = build_schema_digest(bq, BQ_DATASET, ALLOWED_TABLES)
print(f'Schema digest ({len(SCHEMA_DIGEST.splitlines())} tables, {len(SCHEMA_DIGEST)} chars):')
print(SCHEMA_DIGEST or '(empty — make sure 02-bigquery-graph.ipynb has seeded the dataset)')

## Step 3: Generate SQL with Gemini structured output

`response_mime_type='application/json'` + `response_schema` forces the model to return a parseable object. We constrain the dialect (BigQuery Standard SQL), require `SELECT/WITH` only, and ask the model to use backtick-quoted fully-qualified table refs. The execution layer enforces a bytes-billed cap; the model is still asked for a `LIMIT 100` for cheaper round trips.

In [ ]:
_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'sql': {'type': 'string', 'description': 'A single BigQuery Standard SQL SELECT or WITH statement.'},
        'warnings': {
            'type': 'array',
            'items': {'type': 'string'},
            'description': 'Anything the user should know: column not found, ambiguous join, etc.',
        },
    },
    'required': ['sql'],
}


def generate_sql(question: str) -> dict:
    prompt = (
        'You translate analyst questions into BigQuery Standard SQL queries against an OrgIQ dataset.\n'
        '\n'
        'Schema (allow-listed; nothing else exists). Table refs are already fully-qualified — use them verbatim with backticks:\n'
        f'{SCHEMA_DIGEST}\n'
        '\n'
        'Rules:\n'
        '- Output ONE statement. SELECT or WITH only. No INSERT/UPDATE/DELETE/MERGE/DDL.\n'
        '- BigQuery Standard SQL dialect. No trailing semicolon.\n'
        '- Reference tables with backticked fully-qualified names exactly as shown above (e.g., `project.dataset.table`).\n'
        '- Only reference tables and columns from the schema above.\n'
        '- Prefer explicit joins and aliases; never use SELECT *.\n'
        '- Cap result sets at 100 rows with LIMIT 100.\n'
        '- If the question is ambiguous, add a warning string; still emit your best-effort SQL.\n'
        '\n'
        f'Question: {question}\n'
    )
    resp = _client.models.generate_content(
        model=TEXT2SQL_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=1024,
            response_mime_type='application/json',
            response_schema=RESPONSE_SCHEMA,
        ),
    )
    # When response_schema is set, .text is the JSON body.
    return json.loads(resp.text)


QUESTION = 'Who are the top five people by number of skills, and what is their department?'
candidate = generate_sql(QUESTION)
print('Generated candidate:')
print(json.dumps(candidate, indent=2))

## Step 4: Validate before executing

Three layers of static checks before we even ask BigQuery to dry-run:

1. Single statement only — no stacked queries.
2. Starts with `SELECT` or `WITH` — and no forbidden DDL/DML keywords anywhere (`INSERT`, `UPDATE`, `DELETE`, `MERGE`, `TRUNCATE`, `CREATE`, `DROP`, `ALTER`, `GRANT`, `REVOKE`).
3. Every table reference (after stripping backticks) ends in an allow-listed table name.

BigQuery's IAM check is the real boundary in production — the validator just catches obvious badness before we burn a job-slot.

In [ ]:
import re

FORBIDDEN = re.compile(
    r'\b(INSERT|UPDATE|DELETE|MERGE|TRUNCATE|CREATE|DROP|ALTER|GRANT|REVOKE|COPY|CALL|EXPORT|LOAD)\b',
    re.IGNORECASE,
)


class SqlValidationError(Exception):
    pass


def _strip_table_ref(ref: str) -> str:
    """Pull the bare table name out of a (possibly backticked, possibly qualified) ref."""
    ref = ref.strip().strip('`')
    # `project.dataset.table` or `dataset.table` or `table` — take the last segment.
    return ref.split('.')[-1].lower()


def validate_sql(sql: str, allowed_tables: list[str]) -> str:
    stripped = sql.strip().rstrip(';').strip()
    if ';' in stripped:
        raise SqlValidationError('multiple statements detected — only one query permitted')
    head = stripped.lstrip('(').split(None, 1)[0].upper()
    if head not in ('SELECT', 'WITH'):
        raise SqlValidationError(f'statement must start with SELECT or WITH, got {head!r}')
    if FORBIDDEN.search(stripped):
        raise SqlValidationError('statement contains a forbidden DDL/DML keyword')
    allowed = {t.lower() for t in allowed_tables}
    # Collect identifiers that follow FROM/JOIN, with or without backticks.
    refs = re.findall(
        r'\b(?:from|join)\s+(`[^`]+`|[a-zA-Z_][a-zA-Z0-9_\.]*)',
        stripped,
        re.IGNORECASE,
    )
    referenced = {_strip_table_ref(r) for r in refs}
    illegal = referenced - allowed
    if illegal:
        raise SqlValidationError(f'references disallowed tables: {sorted(illegal)}')
    if not referenced:
        raise SqlValidationError('no allow-listed tables referenced')
    return stripped


try:
    safe_sql = validate_sql(candidate['sql'], ALLOWED_TABLES)
    print('✓ SQL passed static validation\n')
    print(safe_sql)
    if candidate.get('warnings'):
        print('\nModel warnings:')
        for w in candidate['warnings']:
            print(f'  - {w}')
except SqlValidationError as exc:
    print(f'✗ Validation failed: {exc}')
    raise

## Step 5: Dry-run + execute against BigQuery

Two more layers before we look at any rows:

1. **Dry-run guard** — `dry_run=True, use_query_cache=False` makes BigQuery plan the query and return `total_bytes_processed` without billing. We reject anything that would scan more than `DRY_RUN_BYTES_CAP` (default 100 MB) — that catches the model writing a full-table cartesian by accident.
2. **Bytes-billed cap** — `maximum_bytes_billed=EXEC_BYTES_BILLED_CAP` (default 10 MB) is the hard runtime ceiling. If BigQuery exceeds it during the real run it raises `Forbidden(reason='bytesBilledLimitExceeded')` and the job is killed — we catch and re-raise as a clean error.

In [ ]:
from google.api_core import exceptions as gax_exceptions


def dry_run_estimate(sql: str) -> int:
    """Plan the query without billing — returns total bytes that would be processed."""
    dry = bq.query(
        sql,
        job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
    )
    return dry.total_bytes_processed


def run_select(sql: str, row_cap: int = 100) -> list[dict]:
    estimate = dry_run_estimate(sql)
    print(f'Dry-run: would scan {estimate:,} bytes')
    if estimate > DRY_RUN_BYTES_CAP:
        raise ValueError(
            f'query would scan {estimate:,} bytes (> cap {DRY_RUN_BYTES_CAP:,}); refusing to execute'
        )
    job_config = bigquery.QueryJobConfig(maximum_bytes_billed=EXEC_BYTES_BILLED_CAP)
    try:
        df = bq.query(sql, job_config=job_config).to_dataframe()
    except gax_exceptions.Forbidden as exc:
        raise RuntimeError(f'BigQuery refused the query (likely bytesBilledLimitExceeded): {exc}') from exc
    return df.head(row_cap).to_dict(orient='records')


rows = run_select(safe_sql)
print(f'✓ {len(rows)} row(s) returned')
print(json.dumps(rows[:10], indent=2, default=str))

## Step 6: Surface a grounded answer (cheap model)

Hand the validated rows to `gemini-2.5-flash` — the strong reasoning model already did the work in Step 3, all we need now is a one-paragraph natural-language summary. Splitting this off from the SQL step (a) is cheaper and (b) makes the grounding visible: the summary cell sees only validated rows, not the raw user question paired with a SQL prompt.

In [ ]:
def summarise(question: str, rows: list[dict]) -> str:
    prompt = (
        'Answer the analyst question using ONLY the rows below. If the rows do not\n'
        'support a clean answer, say so. Keep it under three sentences.\n'
        '\n'
        f'Question: {question}\n'
        f'Rows (JSON): {json.dumps(rows[:20], default=str)}\n'
    )
    resp = _client.models.generate_content(
        model=GROUNDING_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(temperature=0.2, max_output_tokens=300),
    )
    return resp.text or ''


answer = summarise(QUESTION, rows)
print('Answer:\n')
print(answer)

## Step 7: Adversarial cases the validator must reject

Sanity check: injection attempts, DDL, and catalog-snooping must never reach BigQuery. The BQ-specific test (`DROP SCHEMA orgiq`) is the new addition over the AlloyDB version — same SELECT-only validator catches it via the forbidden-keyword regex.

In [ ]:
ADVERSARIAL_CASES = [
    'SELECT 1; DROP TABLE people;',                                           # stacked statement
    'DELETE FROM `orgiq.people` WHERE 1=1;',                                  # DML
    f'DROP SCHEMA `{PROJECT_ID}.{BQ_DATASET}` CASCADE;',                       # BQ-specific DDL
    'MERGE INTO `orgiq.people` USING (SELECT 1 AS id) s ON TRUE WHEN MATCHED THEN DELETE',  # MERGE
    f'SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.INFORMATION_SCHEMA.TABLES`',    # catalog snoop (not in allow-list)
]
for sql in ADVERSARIAL_CASES:
    try:
        validate_sql(sql, ALLOWED_TABLES)
        print(f'✗ accepted (should have rejected): {sql}')
    except SqlValidationError as exc:
        print(f'✓ rejected: {sql!r:80s} → {exc}')

## Summary

- Schema digest built from `INFORMATION_SCHEMA.COLUMNS` with a parameterised allow-list
- Gemini 2.5 Pro generated SQL via `response_schema` (typed JSON `{sql, warnings}`)
- Validator enforces SELECT/WITH-only, no DDL/DML/MERGE, allow-listed tables, handles backticked refs
- Dry-run guard rejects queries that would scan more than `DRY_RUN_BYTES_CAP` (default 100 MB)
- Execution rides `maximum_bytes_billed` for a hard runtime ceiling
- Grounded summary uses `gemini-2.5-flash` over validated rows
- Adversarial cases (stacked statements, DDL, MERGE, catalog snooping) confirmed rejected

**Open question / carry-over from AlloyDB:** the AlloyDB version relied on a dedicated `pg_read_all_data` read-only role as the second line of defence. The BigQuery analogue is a service account with only `roles/bigquery.dataViewer` + `roles/bigquery.jobUser` (or table-scoped `getData`) — that's an IAM/infra task, not a notebook one. ADC is fine for the notebook; bake the service account in for any deployed surface.

**Next:** `19-cleanup.ipynb`.